# `canonicalKDTree` Performance Study

This notebook benchmarks the `canonicalKDTree` method — a canonical distributed KD-tree construction that uses Spark's built-in `percentile_approx` to find splitting medians — across combinations of dataset size, approximation accuracy parameter `accuracy`, data distribution, and tree depth. Results (runtime and leaf-balance precision) are collected into a Pandas DataFrame and written to S3 as Parquet.

**Source module:** `canonicalKDTree.py` (loaded via `spark.sparkContext.addPyFile`).  
**Public API used in this notebook:**

| Function / method | Attached to | Purpose |
|---|---|---|
| `DataFrame.canonicalKDTree(variables, depth, accuracy)` | `pyspark.sql.DataFrame` | Build a KD-tree using `percentile_approx` for median estimation |
| `DataFrame.treePrecision(tree)` | `pyspark.sql.DataFrame` | Compute a leaf-balance precision score for an already-built tree |


## 1. Register the module with the Spark cluster

`addPyFile` ships `canonicalKDTree.py` from `s3://jcgs/code/` to every executor in the cluster. This makes the UDFs and helper functions defined in the module — including `updateleaf` and `f_assign_leaf` — available on all workers before any distributed map step is triggered.


In [ ]:
spark.sparkContext.addPyFile("s3://jcgs/code/canonicalKDTree.py")

## 2. Imports

- **`numpy`** — used internally by `canonicalKDTree.py` to store splitting points as NumPy arrays in the tree structure.
- **`pandas`** — used to accumulate per-run performance records into a summary table and to write results to S3.
- **`time`** — used to measure wall-clock elapsed time around each `canonicalKDTree` call.
- **`canonicalKDTree *`** — wildcard import that monkey-patches `pyspark.sql.DataFrame` with the methods `canonicalKDTree`, `treeLeafCounts`, and `treePrecision`.


In [ ]:
import numpy
import pandas
import time
from canonicalKDTree import *

## 3. Experimental grid

The benchmark study is defined by five configuration variables spanning four independent experimental factors.

| Variable | Values | Meaning |
|---|---|---|
| `variables` | `['x', 'y']` | Column names used as the two splitting axes for the KD-tree |
| `size_list` | `[26, 27, 28, 29, 30]` | Dataset size as a power-of-2 exponent; each entry `s` corresponds to $2^s$ rows |
| `parameter_list` | `[10, 100, 1000, 10000]` | Values of `accuracy`, the accuracy parameter passed to Spark's `percentile_approx`; higher values yield a more accurate median estimate at greater computational cost |
| `distribution_list` | `['blobs', 'normal']` | Synthetic data distributions whose Parquet datasets are pre-stored on S3 |
| `depth_list` | `[4, 6, 8, 10]` | KD-tree depth values (number of recursive splitting levels) |

The full factorial grid across the four factors (`size_list` × `parameter_list` × `distribution_list` × `depth_list`) yields $5 \times 4 \times 2 \times 4 = 160$ benchmark combinations.


In [ ]:
variables = ['x', 'y']
size_list = [26, 27, 28, 29, 30]
parameter_list = [10, 100, 1000, 10000]
distribution_list = ['blobs', 'normal']
depth_list = [4, 6, 8, 10]

## 4. Benchmark loop

The four nested loops iterate over `size_list`, `parameter_list`, `distribution_list`, and `depth_list`. For each combination of `(size, accuracy, distribution, depth)`, the following steps are executed.

1. **Load data** — reads a pre-generated Parquet dataset from S3 at the path  
   `s3://jcgs/data/{distribution}/2^{size}/data.parquet/`  
   and repartitions it to 5,000 partitions.

2. **Build the tree** — calls `data.canonicalKDTree(variables, depth=depth, accuracy=accuracy)`, which internally:
   - Adds a `leaf` column initialised to `0` via `withColumn`.
   - Cycles through `variables` (i.e. `['x', 'y']`) repeatedly for `depth` levels using `itertools.cycle` and `islice`.
   - At each level calls `growBranch(data, depth, variable_axis, accuracy)`, which:
     - Groups by the current `leaf` value and computes the per-leaf median of `variable_axis` using `percentile_approx(..., 0.5, lit(accuracy))`.
     - Collects the ordered list of medians to the driver.
     - Maps every row through `updateleaf`, which doubles the current leaf index and increments by 1 if the row's value exceeds its leaf's median, thereby routing the row left (even index) or right (odd index).
   - Appends a dict `{'Depth': depth, 'Splitting variable': variable_axis, 'Splitting points': numpy.array(medians)}` to the tree list at each level.
   - Returns the complete tree as a list of `depth` such dicts in BFS level order.

3. **Time the build** — wall-clock elapsed time in seconds is captured with `time.time()` bracketing the `canonicalKDTree` call and stored as `runtime`.

4. **Evaluate precision** — calls `data.treePrecision(tree)`, which assigns every row to its leaf by traversing the tree from depth 0, counts rows per leaf, and returns
   $$\text{precision} = -\ln\!\left(\frac{\sum_\ell |C_\ell - \bar C|}{\sum_\ell C_\ell}\right)$$
   where $C_\ell$ is the row count in leaf $\ell$ and $\bar C$ is the mean leaf count. Higher values indicate better balance.

5. **Record results** — constructs a dict `performance` with keys `depth`, `precision`, `runtime`, `distribution`, `parameter`, `size`, and `algorithm` (fixed to `"canonical"`), then appends it to `performance_list`.


In [ ]:
for size in size_list:
    for accuracy in parameter_list:
        for distribution in distribution_list:
            for depth in depth_list:
                data = spark.read.parquet(f's3://jcgs/data/{distribution}/2^{size}/data.parquet/').repartition(5000)
                start = time.time()
                tree = data.canonicalKDTree(variables, depth = depth, accuracy = accuracy)
                stop = time.time()
                runtime = stop - start
                precision = data.treePrecision(tree)
                performance = {'depth': depth, 'precision': precision, 'runtime': runtime}
                performance['distribution'] = distribution
                performance['parameter'] = f"accuracy = {accuracy}"
                performance['size'] = f"2^{size}"
                performance['algorithm'] = "canonical"
                performance_list.append(performance)            

## 5. Collect and write results

Constructs a Pandas DataFrame from `performance_list` (a list of dicts) and writes it to S3 as a Parquet file at `s3://jcgs/output/canonicalKDTree_performance.parquet`.

The output has one row per `(size, accuracy, distribution, depth)` combination with columns: `depth`, `precision`, `runtime`, `distribution`, `parameter`, `size`, and `algorithm`.


In [ ]:
performance = pandas.DataFrame(performance_list)
performance.to_parquet("s3://jcgs/output/canonicalKDTree_performance.parquet")